# 03 Modelo Clasificacion Ticket Alto

Este notebook trabaja la etapa de clasificacion de tickets altos usando **Regresion Logistica**.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parents[1]
PARQUET_PATH = PROJECT_ROOT / "parquets" / "02_EDA_Base_Tickets" / "02_base_eda_tickets.parquet"
df = pd.read_parquet(PARQUET_PATH)
df.head()

In [ ]:
features = [
    "dia", "mes", "trimestre", "dia_semana", "dia_tipo", "fin_semana",
    "ciudad", "capacidad_sucursal", "tipo_empleado", "salario", "turno",
    "numero_mesa", "capacidad_mesa", "metodo_pago", "lineas_ticket",
    "cantidad_total", "platillos_distintos", "categorias_distintas",
    "incluye_bebida", "incluye_postre", "incluye_entrada", "incluye_platillo_fuerte"
]
target = "ticket_alto"

X = df[features].copy()
y = df[target].copy()
X.shape, y.shape

In [ ]:
numericas = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categoricas = X.select_dtypes(include=["object"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numericas
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categoricas
        )
    ]
)

modelo = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced"))
])
modelo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
len(X_train), len(X_test)

In [ ]:
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:, 1]

In [ ]:
metricas = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_prob)
}
metricas

In [ ]:
confusion_matrix(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))